# Step 5: Fast Local Qdrant Ingestion (CPU Optimized)

This notebook generates embeddings for 4,000+ LeetCode problems and pushes them to your Qdrant cluster entirely on your local CPU.

**Optimizations:**
- Model: `BAAI/bge-small-en-v1.5` (Extremely fast, high quality for retrieval, 384 dimensions)
- Backend: `ONNX Runtime` (Provides a 2x-5x speedup on Intel CPUs without needing a dedicated ML GPU)
- Batch Size: `64`

**Required:** A `.env` file in this folder with `QDRANT_URL` and `QDRANT_API_KEY`.

In [1]:
# Cell 1 — Install Dependencies
# Note: optimum[onnxruntime] provides the C++ acceleration backend for the CPU
!pip install -q qdrant-client sentence-transformers "optimum[onnxruntime]" beautifulsoup4 pandas python-dotenv


In [2]:
# Cell 2 — Load Secrets & Initialize Qdrant
import os
import json
from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

load_dotenv()

QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_KEY = os.environ.get("QDRANT_API_KEY")

assert QDRANT_URL and QDRANT_KEY, "Missing QDRANT credentials in .env file!"

print("Initializing Qdrant Client...")
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY)
print("✅ Qdrant Client initialized.")

Initializing Qdrant Client...
✅ Qdrant Client initialized.


In [3]:
# Cell 3 — Load JSONL Data
PROJECT_ROOT = Path(r"C:/PRAKSHIT/VS CODE/Prep Assistant Project/placed_in")
LC_FILE = PROJECT_ROOT / "data" / "my" / "leetcode_clean.jsonl"

if not LC_FILE.exists():
    raise FileNotFoundError(f"Could not find '{LC_FILE}'. Run data cleaning first!")

lc_records = []
with LC_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            lc_records.append(json.loads(line))

print(f"✅ Loaded {len(lc_records)} problems from {LC_FILE.name}")

✅ Loaded 3943 problems from leetcode_clean.jsonl


In [4]:
# Cell 4 — Setup Qdrant Collection for BAAI/bge-small-en-v1.5
COLLECTION_NAME = "leetcode_problems"
VECTOR_SIZE = 384 # bge-small-en-v1.5 native dimensionality

collections = qdrant.get_collections().collections
collection_names = [c.name for c in collections]

if COLLECTION_NAME in collection_names:
    print(f"Collection '{COLLECTION_NAME}' already exists. Deleting it to ensure clean state and correct dimensions ({VECTOR_SIZE})...")
    qdrant.delete_collection(collection_name=COLLECTION_NAME)

print(f"Creating Qdrant collection '{COLLECTION_NAME}' with size={VECTOR_SIZE}...")
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)
print("✅ Qdrant collection created.")

Creating Qdrant collection 'leetcode_problems' with size=384...
✅ Qdrant collection created.


In [5]:
# Cell 5 — Hyper-Fast CPU Embeddings via ONNX
print("Loading BAAI/bge-small-en-v1.5 model with ONNX Runtime acceleration...")
# Note the backend='onnx' which gives us the massive CPU speedup
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    backend="onnx",
    device="cpu"
)
print("✅ ONNX Accelerated Model loaded successfully.")

def clean_html(raw_html):
    if not raw_html:
        return ""
    return BeautifulSoup(str(raw_html), "html.parser").get_text(separator=" ").strip()

batch_size = 64
total = len(lc_records)
print(f"Starting ONNX CPU Embeddings & Qdrant Upsert for {total} problems...")
print("This should take under a minute on a modern i5 CPU.\n")

for i in range(0, total, batch_size):
    batch = lc_records[i:i+batch_size]
    texts_to_embed = []
    points = []
    
    # 1. Prepare semantic texts
    for record in batch:
        clean_text = clean_html(record.get("content", ""))
        # Use the camelCase 'topicTags' key from the raw jsonl
        tags = ", ".join(record.get("topicTags", [])) 
        semantic_text = f"Title: {record.get('title', '')}\nTags: {tags}\n\nProblem Statement:\n{clean_text}"
        texts_to_embed.append(semantic_text)
        
    # 2. Generate Embeddings (Extremely fast via ONNX)
    embeddings = model.encode(texts_to_embed, show_progress_bar=False, convert_to_numpy=True)
    
    # 3. Prepare Qdrant Points
    for idx, record in enumerate(batch):
        payload = {
            "id": record["id"],
            "title": record["title"],
            "slug": record["slug"],
            "difficulty": record["difficulty"],
            "category": record["category"],
            "topic_tags": record.get("topicTags", [])
        }
        
        points.append(
            PointStruct(id=record["id"], vector=embeddings[idx].tolist(), payload=payload)
        )
        
    # 4. Upsert to Qdrant via API
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"  ✅ Encoded & upserted batch {i//batch_size + 1} ({min(i+batch_size, total)}/{total})")

print("\n🎉 Vector embeddings successfully computed via ONNX and ingested into Qdrant!")

Loading BAAI/bge-small-en-v1.5 model with ONNX Runtime acceleration...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\sprak\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sprak\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ONNX Accelerated Model loaded successfully.
Starting ONNX CPU Embeddings & Qdrant Upsert for 3943 problems...
This should take under a minute on a modern i5 CPU.

  ✅ Encoded & upserted batch 1 (64/3943)
  ✅ Encoded & upserted batch 2 (128/3943)
  ✅ Encoded & upserted batch 3 (192/3943)
  ✅ Encoded & upserted batch 4 (256/3943)
  ✅ Encoded & upserted batch 5 (320/3943)
  ✅ Encoded & upserted batch 6 (384/3943)
  ✅ Encoded & upserted batch 7 (448/3943)
  ✅ Encoded & upserted batch 8 (512/3943)
  ✅ Encoded & upserted batch 9 (576/3943)
  ✅ Encoded & upserted batch 10 (640/3943)
  ✅ Encoded & upserted batch 11 (704/3943)
  ✅ Encoded & upserted batch 12 (768/3943)
  ✅ Encoded & upserted batch 13 (832/3943)
  ✅ Encoded & upserted batch 14 (896/3943)
  ✅ Encoded & upserted batch 15 (960/3943)
  ✅ Encoded & upserted batch 16 (1024/3943)
  ✅ Encoded & upserted batch 17 (1088/3943)
  ✅ Encoded & upserted batch 18 (1152/3943)
  ✅ Encoded & upserted batch 19 (1216/3943)
  ✅ Encoded & upserted b